# ELEC5622 Project 2 | HEp-2 classification

This section imports the Python libraries, checks which accelerator is available (CUDA, Apple MPS or CPU) and sets up the folders used to store the results and loads the datasets.


## 1. Check local environment and configure persistent folders

In [ ]:
import os, io, csv, gc, json, time, copy, random, shutil, zipfile, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.transforms import InterpolationMode

SEED = 5622
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                      'mps' if torch.backends.mps.is_available() else 'cpu')
print('Machine:', platform.machine(), '| PyTorch:', torch.__version__, '| device:', DEVICE)
if DEVICE.type == 'cpu':
    print('WARNING: No accelerated backend detected. Full training will be slow on CPU.')


PROJECT_FOLDER = Path.cwd().resolve()
DATA_DIR = PROJECT_FOLDER / 'Three Datasets for Training, Validation and Test'
LABEL_CSV = PROJECT_FOLDER / 'gt_training.csv'
WORK = PROJECT_FOLDER / 'ELEC5622_Project2'
RESULTS_DIR = WORK / 'results'

assert LABEL_CSV.is_file(), f'Missing label file: {LABEL_CSV}'
assert DATA_DIR.is_dir(), f'Missing dataset folder: {DATA_DIR}'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Labels:', LABEL_CSV)
print('Dataset folder:', DATA_DIR)
print('Persistent results folder:', RESULTS_DIR)


### Load the dataset

Here I load the images and match them with their labels from the CSV file. I also check the number of images in each dataset and make sure there are no duplicate image IDs.

## 2. Unpack nested datasets and build reliable label tables

In [ ]:
SPLIT_SIZES = {'training': 8701, 'validation': 2175, 'test': 2720}
label_table = pd.read_csv(LABEL_CSV)
for split, expected_count in SPLIT_SIZES.items():
    split_path = DATA_DIR / split
    actual_count = len(list(split_path.glob('*.png')))
    assert actual_count == expected_count, (
        f'{split}: expected {expected_count} PNG files at {split_path}, found {actual_count}'
    )
    print(f'{split}: {actual_count} images OK')

assert label_table.columns.tolist() == ['Image ID', 'Image class']
assert len(label_table) == 13596 and label_table['Image ID'].is_unique
ID_TO_LABEL = dict(zip(label_table['Image ID'].astype(int), label_table['Image class'].astype(str)))
CLASSES = sorted(label_table['Image class'].unique().tolist())
CLASS_TO_INDEX = {name: i for i, name in enumerate(CLASSES)}
assert len(CLASSES) == 6

split_tables = {}
all_ids = []
for split, expected_count in SPLIT_SIZES.items():
    rows = []
    for img_path in sorted((DATA_DIR / split).glob('*.png')):
        image_id = int(img_path.stem)
        rows.append({'path': str(img_path), 'image_id': image_id,
                     'class_name': ID_TO_LABEL[image_id],
                     'label': CLASS_TO_INDEX[ID_TO_LABEL[image_id]]})
        all_ids.append(image_id)
    split_tables[split] = pd.DataFrame(rows)
    assert len(rows) == expected_count
assert len(all_ids) == len(set(all_ids)) == 13596
print('Classes:', CLASSES)
print(pd.DataFrame({split: df['class_name'].value_counts().reindex(CLASSES, fill_value=0)
                    for split, df in split_tables.items()}))

## 3. View actual training examples
I displayed one example from each of the six classes to check what the images look like and whether the labels are correct.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for ax, cls in zip(axes.ravel(), CLASSES):
    path = split_tables['training'].query('class_name == @cls').iloc[0]['path']
    with Image.open(path) as im:
        ax.imshow(im.convert('L'), cmap='gray')
    ax.set_title(cls if cls != 'NuMem' else 'NuMem (Nuclear Membrane)')
    ax.axis('off')
fig.suptitle('Examples from the tutor-provided HEp-2 training dataset')
fig.tight_layout()
plt.show()

## 4. Dataset transforms and DataLoaders
The images are resized to 224 × 224 and converted into three channels for the pretrained networks. I also added random rotations and flips to the training images to help the models learn different orientations.

In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 8 if DEVICE.type == 'mps' else (32 if DEVICE.type == 'cuda' else 8)
NUM_WORKERS = 2 if DEVICE.type == 'cuda' else 0
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Rotations preserve HEp-2 class labels.
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(degrees=180, interpolation=InterpolationMode.BILINEAR, fill=0),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class CellDataset(Dataset):
    def __init__(self, records, transform):
        self.records = records.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.records)
    def __getitem__(self, ix):
        row = self.records.iloc[ix]
        with Image.open(row['path']) as im:
            # Grayscale -> 3 identical channels for ImageNet-pretrained models
            image = im.convert('RGB')
        return self.transform(image), int(row['label'])

train_ds = CellDataset(split_tables['training'], train_transform)
val_ds = CellDataset(split_tables['validation'], val_test_transform)
test_ds = CellDataset(split_tables['test'], val_test_transform)

def make_loader(ds, shuffle=False, seed=SEED):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'),
                      generator=g)

val_loader = make_loader(val_ds)
test_loader = make_loader(test_ds)
images, labels = next(iter(make_loader(train_ds, shuffle=True)))
assert images.shape[1:] == (3, IMAGE_SIZE, IMAGE_SIZE)
assert labels.min() >= 0 and labels.max() <= 5
print('Training batch:', tuple(images.shape), '| labels:', tuple(labels.shape))

## 5. CNN architectures and freezing strategies
This section sets up AlexNet and ResNet34 with six output classes. For partial fine-tuning, only selected layers are trained, whereas full fine-tuning updates the whole network.

In [ ]:
def build_model(architecture, strategy, pretrained=True):
    if architecture == 'AlexNet':
        weights = models.AlexNet_Weights.IMAGENET1K_V1 if pretrained else None
        net = models.alexnet(weights=weights)
        in_features = net.classifier[6].in_features
        net.classifier[6] = nn.Linear(in_features, len(CLASSES))
        if strategy == 'partial':
            # Train FC6, FC7, FC8; freeze AlexNet convolutional feature extractor.
            for p in net.parameters(): p.requires_grad = False
            for p in net.classifier.parameters(): p.requires_grad = True
    elif architecture == 'ResNet34':
        weights = models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        net = models.resnet34(weights=weights)
        net.fc = nn.Linear(net.fc.in_features, len(CLASSES))
        if strategy == 'partial':
            for p in net.parameters(): p.requires_grad = False
            for p in net.layer4.parameters(): p.requires_grad = True
            for p in net.fc.parameters(): p.requires_grad = True
    else:
        raise ValueError('Unknown architecture: '+str(architecture))
    if strategy not in ['partial','full']:
        raise ValueError('strategy must be partial or full')
    return net

def count_params(net):
    return {'total_parameters': sum(p.numel() for p in net.parameters()),
            'trainable_parameters': sum(p.numel() for p in net.parameters() if p.requires_grad)}

# Avoid downloading four weight sets just to print counts.
for arch in ['AlexNet', 'ResNet34']:
    for strategy in ['partial', 'full']:
        net = build_model(arch, strategy, pretrained=False)
        print(f'{arch:8s} {strategy:7s}:', count_params(net))
        del net

## 6. Training and validation helpers
These functions train the models, calculate accuracy and loss, and save the best model based on validation accuracy. Early stopping is used when validation accuracy stops improving.

In [ ]:
LOSS_FN = nn.CrossEntropyLoss()
MAX_EPOCHS = 15
PATIENCE = 4
LR = 1e-4       # Same starting learning rate in all four configurations
WEIGHT_DECAY = 1e-4

def sync_gpu():
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    elif DEVICE.type == 'mps': torch.mps.synchronize()

def freeze_nontrainable_bn(net):
    # Frozen BN layers must not silently change their running statistics.
    for layer in net.modules():
        if isinstance(layer, nn.modules.batchnorm._BatchNorm):
            params = list(layer.parameters())
            if params and not any(p.requires_grad for p in params):
                layer.eval()

def run_epoch(net, loader, optimizer=None):
    training = optimizer is not None
    net.train(training)
    if training:
        freeze_nontrainable_bn(net)
    running_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        if training: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = net(x)
            loss = LOSS_FN(logits, y)
            if training:
                loss.backward()
                optimizer.step()
        n = x.size(0)
        running_loss += loss.item() * n
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += n
    return {'loss': running_loss / total, 'accuracy': correct / total}

def experiment_names():
    return [(arch, strategy) for arch in ['AlexNet', 'ResNet34']
            for strategy in ['partial', 'full']]

def train_experiment(arch, strategy):
    run_name = f'{arch}_{strategy}'
    checkpoint_path = RESULTS_DIR / f'{run_name}_best.pth'
    history_path = RESULTS_DIR / f'{run_name}_history.csv'
    if checkpoint_path.exists() and (RESULTS_DIR / f'{run_name}_metadata.json').exists():
        print('Already trained, skipping:', run_name, '| delete checkpoint to retrain')
        return
    print('\n===', run_name, '===')
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if DEVICE.type == 'cuda': torch.cuda.manual_seed_all(SEED)
    train_loader = make_loader(train_ds, shuffle=True, seed=SEED)
    net = build_model(arch, strategy, pretrained=True).to(DEVICE)
    param_counts = count_params(net)
    optimizer = torch.optim.AdamW((p for p in net.parameters() if p.requires_grad),
                                  lr=LR, weight_decay=WEIGHT_DECAY)
    best_val_acc, best_epoch, no_improve = -1.0, 0, 0
    history, training_seconds = [], 0.0
    for epoch in range(1, MAX_EPOCHS+1):
        sync_gpu(); tic = time.perf_counter()
        tr = run_epoch(net, train_loader, optimizer)
        sync_gpu(); epoch_seconds = time.perf_counter()-tic
        training_seconds += epoch_seconds
        va = run_epoch(net, val_loader)
        history.append({'epoch':epoch, 'train_loss':tr['loss'],
                        'train_accuracy':tr['accuracy'], 'val_loss':va['loss'],
                        'val_accuracy':va['accuracy'], 'training_seconds':epoch_seconds})
        pd.DataFrame(history).to_csv(history_path, index=False)
        print(f'Epoch {epoch:02d} | train {tr["accuracy"]:.4f} | '
              f'val {va["accuracy"]:.4f} | val loss {va["loss"]:.4f} | '
              f'train {epoch_seconds:.1f}s')
        if va['accuracy'] > best_val_acc + 1e-6:
            best_val_acc, best_epoch, no_improve = va['accuracy'], epoch, 0
            torch.save({'model_state_dict': net.state_dict(), 'architecture': arch,
                        'strategy':strategy, 'classes': CLASSES,
                        'best_val_accuracy':best_val_acc, 'best_epoch':best_epoch,
                        'parameters':param_counts}, checkpoint_path)
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print('Early stopping after', epoch, 'epochs')
                break
    metadata = {'architecture':arch, 'strategy':strategy,
                'best_epoch':best_epoch, 'best_val_accuracy':best_val_acc,
                'train_time_seconds':training_seconds, 'epochs_run':len(history),
                'lr':LR, 'weight_decay':WEIGHT_DECAY, 'batch_size': BATCH_SIZE,
                'image_size':IMAGE_SIZE, 'seed':SEED, 'device':str(DEVICE),
                **param_counts}
    (RESULTS_DIR / f'{run_name}_metadata.json').write_text(json.dumps(metadata, indent=2))
    del net, optimizer, train_loader
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    print('Saved:', checkpoint_path)

## 7. Run experiments one at a time

Each call skips a configuration whose checkpoint and metadata already exist in `ELEC5622_Project2/results/`, so re-running this notebook on the machine that trained the models re-evaluates and regenerates figures without retraining.

I trained four configurations using the same learning rate, batch size and maximum number of epochs. This allows me to compare the two architectures and fine-tuning approaches more fairly.

In [ ]:
# Quick model-forward test (downloads the official pretrained AlexNet weights):
probe = build_model('AlexNet', 'partial', pretrained=True).to(DEVICE)
probe.eval()
with torch.no_grad():
    prediction_shape = probe(images[:2].to(DEVICE)).shape
print('Model output shape:', tuple(prediction_shape), 'expected (2, 6)')
assert tuple(prediction_shape) == (2, 6)
del probe
if DEVICE.type == 'cuda': torch.cuda.empty_cache()

In [ ]:
# Existing completed runs (if any)
for arch, strat in experiment_names():
    name = f'{arch}_{strat}'
    print(
        name,
        'SAVED' if (RESULTS_DIR / f'{name}_best.pth').exists()
        and (RESULTS_DIR / f'{name}_metadata.json').exists()
        else 'NOT YET'
    )

# Train AlexNet partial
train_experiment('AlexNet', 'partial')

In [ ]:
# Experiment 2: AlexNet full fine-tuning
train_experiment('AlexNet', 'full')


In [ ]:
# Experiment 3: ResNet34 partial fine-tuning
train_experiment('ResNet34', 'partial')


In [ ]:
# Experiment 4: ResNet34 full fine-tuning
train_experiment('ResNet34', 'full')


## 8. Plot training/validation curves and summarize training resources

Here I plotted the training and validation accuracy for each model. I also compared how many parameters were trained and how long each experiment took.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
rows = []
for ax, (arch, strategy) in zip(axes.ravel(), experiment_names()):
    name = f'{arch}_{strategy}'
    hist_file = RESULTS_DIR / f'{name}_history.csv'
    meta_file = RESULTS_DIR / f'{name}_metadata.json'
    if not hist_file.exists() or not meta_file.exists():
        ax.set_title(f'{name}: not trained yet')
        continue
    hist = pd.read_csv(hist_file)
    meta = json.loads(meta_file.read_text())
    rows.append(meta)
    ax.plot(hist['epoch'], hist['train_accuracy'], label='Training')
    ax.plot(hist['epoch'], hist['val_accuracy'], label='Validation')
    ax.set(xlabel='Epoch', ylabel='Accuracy', title=name)
    ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'all_accuracy_curves.png', dpi=180)
plt.show()
resource_table = pd.DataFrame(rows)
if not resource_table.empty:
    print(resource_table[['architecture','strategy','trainable_parameters',
                          'train_time_seconds','best_epoch','best_val_accuracy']].to_string(index=False))

## 9. Final held-out TEST evaluation

After training, I loaded the best saved version of each model and tested it on 2,720 images. I calculated accuracy, macro F1 and confusion matrices to compare the results.

In [ ]:
@torch.inference_mode()
def predict_all(net, loader):
    net.eval()
    true, pred = [], []
    for x, y in loader:
        logits = net(x.to(DEVICE, non_blocking=True))
        pred.extend(logits.argmax(1).cpu().tolist())
        true.extend(y.tolist())
    return np.asarray(true), np.asarray(pred)

results = []
for arch, strategy in experiment_names():
    name = f'{arch}_{strategy}'
    checkpoint_file = RESULTS_DIR / f'{name}_best.pth'
    meta_file = RESULTS_DIR / f'{name}_metadata.json'
    assert checkpoint_file.exists() and meta_file.exists(), f'Missing trained experiment: {name}'
    checkpoint = torch.load(checkpoint_file, map_location='cpu', weights_only=True)
    assert checkpoint['classes'] == CLASSES
    net = build_model(arch, strategy, pretrained=False).to(DEVICE)
    net.load_state_dict(checkpoint['model_state_dict'])
    y_true, y_pred = predict_all(net, test_loader)
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    meta = json.loads(meta_file.read_text())
    results.append({'architecture':arch, 'strategy':strategy,
                    'test_accuracy':acc, 'test_macro_f1':macro_f1,
                    'best_val_accuracy':meta['best_val_accuracy'],
                    'trainable_parameters':meta['trainable_parameters'],
                    'training_seconds':meta['train_time_seconds']})
    report = classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True,
                                   zero_division=0)
    pd.DataFrame(report).T.to_csv(RESULTS_DIR / f'{name}_classification_report.csv')
    cm = confusion_matrix(y_true, y_pred)
    pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(RESULTS_DIR / f'{name}_confusion_matrix.csv')
    fig, ax = plt.subplots(figsize=(8, 7))
    ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, colorbar=False,
                                                             xticks_rotation=40)
    ax.set_title(name + ' (test)')
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / f'{name}_confusion_matrix.png', dpi=180)
    plt.show()
    print(f'{name}: test accuracy={acc:.4f} | macro F1={macro_f1:.4f}')
    del net
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

comparison = pd.DataFrame(results)
comparison.to_csv(RESULTS_DIR / 'final_comparison.csv', index=False)
print(comparison.to_string(index=False))

## 10. Regenerate report figures

Figures 1, 4, 5 and 7 of the report are produced here from the saved label table and result CSVs, so every figure in the report is reproducible from code.

In [ ]:
from decimal import Decimal, ROUND_HALF_UP

NAVY = '#19324D'
LABELS = {'AlexNet_partial': 'AlexNet Partial', 'AlexNet_full': 'AlexNet Full',
          'ResNet34_partial': 'ResNet34 Partial', 'ResNet34_full': 'ResNet34 Full'}
SHORT = {'Centromere': 'Cent.', 'Golgi': 'Golgi', 'Homogeneous': 'Homog.',
         'NuMem': 'NuMem', 'Nucleolar': 'Nucle.', 'Speckled': 'Speck.'}

def pct(x):
    # Round half up so 0.93125 is reported as 93.13 (matches the report text).
    return str(Decimal(repr(round(x * 100, 6))).quantize(Decimal('0.01'), ROUND_HALF_UP))

def style(ax, title):
    ax.set_title(title, loc='left', fontsize=15, fontweight='bold', color=NAVY)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', color='#E6EAF0', linewidth=0.8)
    ax.set_axisbelow(True)

def fig_class_counts(train_table, out):
    counts = train_table['class_name'].value_counts().reindex(CLASSES)
    fig, ax = plt.subplots(figsize=(10.8, 3.7))
    bars = ax.bar(counts.index, counts.values, color='#3B6FAE')
    ax.bar_label(bars, labels=[f'{v:,}' for v in counts.values], fontsize=10, color=NAVY)
    ax.set_ylabel('Training images')
    style(ax, 'Training-set class distribution')
    fig.tight_layout(); fig.savefig(out, dpi=180); plt.show()

def fig_val_test(comparison, out):
    names = [f'{a}_{s}' for a, s in zip(comparison['architecture'], comparison['strategy'])]
    x = np.arange(len(names)); w = 0.36
    fig, ax = plt.subplots(figsize=(10.8, 3.9))
    for offset, col, colour, label in [(-w / 2, 'best_val_accuracy', '#A3B7C9', 'Best validation'),
                                       (w / 2, 'test_accuracy', '#13818A', 'Held-out test')]:
        bars = ax.bar(x + offset, comparison[col] * 100, w, color=colour, label=label)
        ax.bar_label(bars, labels=[pct(v) for v in comparison[col]], fontsize=9, padding=2)
    ax.set_xticks(x, [LABELS[n].replace(' ', '\n') for n in names])
    ax.set_ylim(88, 101); ax.set_ylabel('Accuracy (%)')
    ax.legend(loc='upper left', ncol=2, frameon=False)
    style(ax, 'Accuracy by architecture and fine-tuning strategy')
    fig.tight_layout(); fig.savefig(out, dpi=180); plt.show()

def fig_tradeoff(comparison, out):
    markers = {'AlexNet_partial': ('o', '#8FA3B1'), 'AlexNet_full': ('s', '#557A95'),
               'ResNet34_partial': ('D', '#6DB5A0'), 'ResNet34_full': ('P', '#13818A')}
    fig, ax = plt.subplots(figsize=(10.8, 3.85))
    for _, r in comparison.iterrows():
        name = f"{r['architecture']}_{r['strategy']}"
        m, c = markers[name]
        minutes, acc = r['training_seconds'] / 60, r['test_accuracy'] * 100
        ax.scatter(minutes, acc, marker=m, color=c, s=170, zorder=3)
        ax.annotate(LABELS[name], (minutes, acc), xytext=(8, 8 if acc > 94.5 else -18),
                    textcoords='offset points', fontsize=10)
    ax.set_xlim(22, 84); ax.set_ylim(92, 99)
    ax.set_xlabel('Cumulative training time (minutes; excludes validation)')
    ax.set_ylabel('Held-out test accuracy (%)')
    ax.grid(axis='x', color='#E6EAF0', linewidth=0.8)
    style(ax, 'Accuracy / training-cost trade-off')
    fig.tight_layout(); fig.savefig(out, dpi=180); plt.show()

def fig_other_cms(results_dir, out, names=('AlexNet_partial', 'AlexNet_full', 'ResNet34_partial')):
    fig, axes = plt.subplots(1, len(names), figsize=(17.6, 6.0))
    for ax, name in zip(axes, names):
        cm = pd.read_csv(results_dir / f'{name}_confusion_matrix.csv', index_col=0)
        ax.imshow(cm.values, cmap='Blues')
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                v = cm.values[i, j]
                ax.text(j, i, v, ha='center', va='center', fontsize=10,
                        color='white' if v > cm.values.max() / 2 else NAVY)
        ticks = [SHORT[c] for c in cm.columns]
        ax.set_xticks(range(len(ticks)), ticks, rotation=45, ha='right')
        ax.set_yticks(range(len(ticks)), ticks)
        ax.set_xlabel('Predicted')
        arch, strategy = name.split('_')
        ax.set_title(f'{arch} - {strategy}', fontsize=15, fontweight='bold', color=NAVY)
    axes[0].set_ylabel('True class')
    fig.tight_layout(); fig.savefig(out, dpi=180); plt.show()


In [ ]:
comparison = pd.read_csv(RESULTS_DIR / 'final_comparison.csv')
fig_class_counts(split_tables['training'], RESULTS_DIR / 'report_fig1_class_counts.png')
fig_val_test(comparison, RESULTS_DIR / 'report_fig4_val_vs_test.png')
fig_tradeoff(comparison, RESULTS_DIR / 'report_fig5_accuracy_vs_time.png')
fig_other_cms(RESULTS_DIR, RESULTS_DIR / 'report_fig7_other_confusion_matrices.png')


## 11. Prepare outputs for the report

This final section saves the graphs, tables and evaluation results so they can be used in the report.

In [ ]:
archive_path = WORK / 'project2_experiment_results.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for ext in ('*.csv', '*.json', '*.png'):
        for file in RESULTS_DIR.glob(ext):
            z.write(file, arcname=file.name)
print('Results archive (charts and metrics, no datasets or checkpoints):', archive_path)